# Ep. 03 — Conversation memory, context window, and sliding window

Technical notebook: four progressions. Each request is independent unless **you** send history.


## Setup

Shared helper: real API if `OPENAI_API_KEY` is set, otherwise offline mock (including context length exceeded).


In [ ]:
import os
import json
from typing import Any

# --- Vanilla chat helper: real OpenAI API if OPENAI_API_KEY is set, else offline mock ---
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USE_REAL = bool(API_KEY)

# Simulated tiny context window for demos when mocking (characters ≈ coarse "tokens")
MOCK_CONTEXT_LIMIT_CHARS = 800


def _approx_tokens(messages: list[dict]) -> int:
    return max(1, sum(len(m.get("content", "")) for m in messages) // 4)


def chat(messages: list[dict], *, max_tokens: int = 200) -> str:
    """Send chat messages. Raises RuntimeError with 'context length exceeded' when over budget."""
    if USE_REAL:
        import urllib.request
        body = json.dumps({
            "model": MODEL,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0,
        }).encode()
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read().decode())
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            text = str(e)
            # Surface provider context errors clearly
            if "context_length" in text.lower() or "maximum context" in text.lower() or "context length" in text.lower():
                raise RuntimeError("context length exceeded: " + text) from e
            # Try to read HTTP error body
            if hasattr(e, "read"):
                try:
                    err_body = e.read().decode()
                    if "context" in err_body.lower() and "length" in err_body.lower():
                        raise RuntimeError("context length exceeded: " + err_body) from e
                except RuntimeError:
                    raise
                except Exception:
                    pass
            raise
    # Offline mock
    total_chars = sum(len(m.get("content", "")) for m in messages)
    if total_chars > MOCK_CONTEXT_LIMIT_CHARS:
        raise RuntimeError(
            f"context length exceeded: mock limit {MOCK_CONTEXT_LIMIT_CHARS} chars, got {total_chars}"
        )
    # Find last user message and any earlier user facts
    users = [m["content"] for m in messages if m.get("role") == "user"]
    if not users:
        return "(no user message)"
    last = users[-1]
    # If history includes a prior fact about a secret/codeword, recall it
    history_text = "\n".join(users[:-1])
    if "lembra" in last.lower() or "remember" in last.lower() or "recuerd" in last.lower() or "qual" in last.lower() or "what" in last.lower() or "cuál" in last.lower() or "cual" in last.lower():
        # Look for CODEWORD / palavra / palabra patterns in earlier turns
        for u in users[:-1]:
            for marker in ("CODEWORD:", "palavra-chave:", "palabra clave:", "secret:", "segredo:", "secreto:"):
                if marker.lower() in u.lower():
                    # extract after marker
                    idx = u.lower().index(marker.lower())
                    val = u[idx + len(marker):].strip().split()[0]
                    return f"Recalled from history: {val}"
        if len(users) == 1:
            return "I have no prior messages in this request, so I don't know."
        return "I don't see that fact in the provided history."
    if len(users) == 1:
        return f"Got it. (stateless ack) Stored nothing server-side."
    return f"OK. History turns={len(users)}. Last: {last[:80]}"


def sliding_window(messages: list[dict], max_messages: int) -> list[dict]:
    """Keep system (if any) + the most recent messages up to max_messages total dialog slots."""
    if not messages:
        return []
    system = [m for m in messages if m.get("role") == "system"]
    dialog = [m for m in messages if m.get("role") != "system"]
    keep_dialog = max(0, max_messages - len(system))
    trimmed = dialog[-keep_dialog:] if keep_dialog else []
    return system + trimmed


print("USE_REAL_API =", USE_REAL, "| MODEL =", MODEL if USE_REAL else "(mock)")
print("MOCK_CONTEXT_LIMIT_CHARS =", MOCK_CONTEXT_LIMIT_CHARS)


## 1) The LLM forgets

Two **separate** calls. The second request has no history, so the model cannot recall the fact from the first.


In [ ]:
fact = "CODEWORD: ORBIT-77"
r1 = chat([{"role": "user", "content": f"Remember this secret: {fact}"}])
print("Call 1:", r1)

r2 = chat([{"role": "user", "content": "What is the CODEWORD I told you?"}])
print("Call 2 (no history):", r2)
print("→ Expected: it does NOT know ORBIT-77.")


## 2) Build a history list

Accumulate messages in a list (oldest → newest). Resend the full list each turn so the model can "remember".


In [ ]:
history: list[dict] = []
history.append({"role": "user", "content": "Remember this secret: CODEWORD: ORBIT-77"})
a1 = chat(history)
history.append({"role": "assistant", "content": a1})
print("After turn 1, history len =", len(history))

history.append({"role": "user", "content": "What is the CODEWORD I told you?"})
a2 = chat(history)
history.append({"role": "assistant", "content": a2})
print("Call with history:", a2)
print("→ Expected: recalls ORBIT-77 from the list you sent.")


## 3) Blow the context window

Keep appending until the provider returns **context length exceeded** (real API error if keyed; otherwise a faithful mock).


In [ ]:
big = list(history)  # start from previous history if available
blob = "PADDING " * 50
errored = False
for i in range(1, 40):
    big.append({"role": "user", "content": f"turn-{i}: {blob}"})
    try:
        _ = chat(big, max_tokens=16)
        print(f"ok growth step {i}, messages={len(big)}")
    except RuntimeError as e:
        print("ERROR:", e)
        errored = True
        break
print("Got context length exceeded =", errored)


## 4) Sliding window

When the budget fills, drop the **oldest** dialog messages (keep `system` if present). The model forgets the beginning but keeps the recent thread — and requests stop failing.


In [ ]:
# Rebuild an oversized dialog, then trim
msgs = [{"role": "system", "content": "You are a concise assistant."}]
msgs.append({"role": "user", "content": "Early fact CODEWORD: ORBIT-77 — only said once at the start."})
msgs.append({"role": "assistant", "content": "Ack early fact."})
for i in range(20):
    msgs.append({"role": "user", "content": f"Recent topic chit-chat {i}: " + ("x" * 40)})
    msgs.append({"role": "assistant", "content": f"Ack {i}"})

print("Before trim: messages =", len(msgs), "approx_tokens ≈", _approx_tokens(msgs))
try:
    chat(msgs, max_tokens=16)
    print("Unexpected: oversized call succeeded")
except RuntimeError as e:
    print("Before trim error (expected):", e)

trimmed = sliding_window(msgs, max_messages=8)
print("After sliding_window(max_messages=8):", len(trimmed), "messages")
for m in trimmed:
    print(" ", m["role"], ":", m["content"][:60].replace("\n", " "), "...")

out = chat(trimmed + [{"role": "user", "content": "What is the CODEWORD from the very beginning?"}])
print("Ask about early fact after trim:", out)
print("→ Early CODEWORD likely gone; recent thread still fits.")
out2 = chat(trimmed + [{"role": "user", "content": "What was the last chit-chat index roughly?"}])
print("Ask about recent thread:", out2)
